In [40]:
import re
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive

drive.mount("/content/gdrive")

%cd "/content/gdrive/MyDrive/datasets"

# columns has the name of each column.
df_raw = pd.read_csv('Smartphone_Usage_And_Addiction_Analysis_7500_Rows.csv', delimiter=',', encoding='latin1')
df_raw.head()

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
/content/gdrive/MyDrive/datasets


,transaction_id,user_id,age,gender,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,stress_level,academic_work_impact,addiction_level,addicted_label
0,TXN00001,U00001,21,Male,3.23,2.01,0.89,4.55,7.55,248,154,3.95,Medium,Yes,NaN,0
1,TXN00002,U00002,24,Other,5.09,3.81,2.24,4.44,7.66,127,71,6.71,Medium,Yes,NaN,0
2,TXN00003,U00003,31,Other,6.06,1.36,3.83,2.35,4.92,44,106,8.68,High,No,Mild,0
3,TXN00004,U00004,32,Other,7.83,5.85,1.51,3.54,8.23,178,107,9.77,High,Yes,Moderate,1
4,TXN00005,U00005,25,Male,9.96,5.92,3.42,5.27,6.21,136,177,12.55,Low,No,Severe,1


In [41]:
df = df_raw.copy()

df.drop(columns=["transaction_id"], inplace=True)
df.drop(columns=["user_id"], inplace=True)
df.drop(columns=["addiction_level"], inplace=True)

#df["addiction_level"].fillna("None", inplace=True)

df['academic_work_impact'] = df['academic_work_impact'].map({'Yes': 1, 'No': 0})

df = pd.get_dummies(df, columns=['gender', 'stress_level'], drop_first=False)

df.head()

,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,academic_work_impact,addicted_label,gender_Female,gender_Male,gender_Other,stress_level_High,stress_level_Low,stress_level_Medium
0,21,3.23,2.01,0.89,4.55,7.55,248,154,3.95,1,0,False,True,False,False,False,True
1,24,5.09,3.81,2.24,4.44,7.66,127,71,6.71,1,0,False,False,True,False,False,True
2,31,6.06,1.36,3.83,2.35,4.92,44,106,8.68,0,0,False,False,True,True,False,False
3,32,7.83,5.85,1.51,3.54,8.23,178,107,9.77,1,1,False,False,True,True,False,False
4,25,9.96,5.92,3.42,5.27,6.21,136,177,12.55,0,1,False,True,False,False,True,False


In [42]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

df_train, df_temp = train_test_split(df, test_size=0.2, random_state=42)

df_val, df_test = train_test_split(df_temp, test_size=0.5, random_state=42)

X_train = df_train.drop(columns=["addicted_label"]).copy()
y_train = df_train["addicted_label"]

X_val = df_val.drop(columns=["addicted_label"]).copy()
y_val = df_val["addicted_label"]

X_test = df_test.drop(columns=["addicted_label"]).copy()
y_test = df_test["addicted_label"]

NUMERIC_COLUMNS = [
    "age",
    "daily_screen_time_hours",
    "social_media_hours",
    "gaming_hours",
    "work_study_hours",
    "sleep_hours",
    "notifications_per_day",
    "app_opens_per_day",
    "weekend_screen_time"
]

scaler = StandardScaler()

# fit SOLO con train
X_train[NUMERIC_COLUMNS] = scaler.fit_transform(X_train[NUMERIC_COLUMNS])

# transform con validation y test
X_val[NUMERIC_COLUMNS] = scaler.transform(X_val[NUMERIC_COLUMNS])
X_test[NUMERIC_COLUMNS] = scaler.transform(X_test[NUMERIC_COLUMNS])

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)


X_train shape: (6000, 16)
X_val shape: (750, 16)
X_test shape: (750, 16)


In [43]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

n_features = X_train.shape[1]

model = Sequential([
    Input(shape=(n_features,)),
    Dense(16, activation='relu'),
    Dense(8, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_9 (Dense)                 │ (None, 16)             │           272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 417 (1.63 KB)

 Trainable params: 417 (1.63 KB)

 Non-trainable params: 0 (0.00 B)

In [44]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [45]:
print(X_train.shape)
print(X_train.dtypes)
print(y_train.unique())

(6000, 16)
age                        float64
daily_screen_time_hours    float64
social_media_hours         float64
gaming_hours               float64
work_study_hours           float64
sleep_hours                float64
notifications_per_day      float64
app_opens_per_day          float64
weekend_screen_time        float64
academic_work_impact         int64
gender_Female                 bool
gender_Male                   bool
gender_Other                  bool
stress_level_High             bool
stress_level_Low              bool
stress_level_Medium           bool
dtype: object
[0 1]


In [46]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=32
)

Epoch 1/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.7618 - loss: 0.4850 - val_accuracy: 0.8413 - val_loss: 0.3696
Epoch 2/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8847 - loss: 0.2776 - val_accuracy: 0.8867 - val_loss: 0.2598
Epoch 3/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9000 - loss: 0.2318 - val_accuracy: 0.8933 - val_loss: 0.2375
Epoch 4/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9072 - loss: 0.2117 - val_accuracy: 0.9053 - val_loss: 0.2229
Epoch 5/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9118 - loss: 0.1961 - val_accuracy: 0.9080 - val_loss: 0.2098
Epoch 6/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9172 - loss: 0.1833 - val_accuracy: 0.9133 - val_loss: 0.2013
Epoch 7/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9195 - loss: 0.1729 - val_accuracy: 0.9107 - val_loss: 0.1950
Epoch 8/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9223 - loss: 0.1657 - val_accuracy: 0.

In [47]:
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)
y_pred_prob = model.predict(X_test).ravel()

print("Mínimo:", y_pred_prob.min())
print("Máximo:", y_pred_prob.max())
print("Promedio:", y_pred_prob.mean())
print("Primeras 20 probabilidades:", y_pred_prob[:20])

24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
Mínimo: 0.00020963006
Máximo: 0.99999964
Promedio: 0.7000091
Primeras 20 probabilidades: [0.04171488 0.26368204 0.99910307 0.99996245 0.9986877  0.4824847
 0.99145335 0.9979477  0.9999633  0.99998677 0.6826068  0.83717906
 0.00172494 0.9946523  0.99986416 0.99999726 0.9999403  0.9999942
 0.7781924  0.99994594]


In [48]:
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report, confusion_matrix, accuracy_score

f1 = f1_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy en test:", accuracy)
print("F1-score:", f1)
print("Precision:", precision)
print("Recall:", recall)
print("\nMatriz de confusión:\n", cm)
print("\nReporte completo:\n")
print(classification_report(y_test, y_pred))

Accuracy en test: 0.9133333333333333
F1-score: 0.9383886255924171
Precision: 0.9375
Recall: 0.9392789373814042

Matriz de confusión:
 [[190  33]
 [ 32 495]]

Reporte completo:

              precision    recall  f1-score   support

           0       0.86      0.85      0.85       223
           1       0.94      0.94      0.94       527

    accuracy                           0.91       750
   macro avg       0.90      0.90      0.90       750
weighted avg       0.91      0.91      0.91       750



In [49]:
model.save('best_model2.keras')
joblib.dump(scaler, 'scaler2.pkl')

print("Modelo guardado en: best_model2.keras")
print("Scaler guardado en: scaler2.pkl")


Modelo guardado en: best_model.keras
Scaler guardado en: scaler.pkl
